# 06. Đánh giá agent

Notebook này tải checkpoint tốt nhất từ `05_train` và đo agent kỹ hơn so với lần đánh giá định kỳ.

Nội dung:

1. Nạp các module và cấu hình.
2. Tải agent.
3. Đánh giá cuối: 100 ván, `score_limit` 1.000.
4. Kiểm tra giá trị Q.
5. Xem agent chơi (tùy chọn).

Cách đánh giá theo mục 6 của `Implementation_Notes.md`.

## 1. Nạp các module và cấu hình

Notebook này không cần Experience Replay. `eval_seed` giống seed đánh giá trong `05_train`, để kết quả so sánh được với các lần đánh giá định kỳ.

In [ ]:
%run ./01_environment.ipynb
%run ./04_agent.ipynb

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

EVAL_CONFIG = dict(
    checkpoint_path=os.path.join("checkpoints", "best.weights.h5"),
    hidden_units=256,
    n_episodes=100,
    score_limit=1_000,
    eval_seed=12345,
)

## 2. Tải agent

Mạng được dựng lại bằng cùng kiến trúc rồi nạp trọng số. `hidden_units` phải khớp với cấu hình lúc huấn luyện.

In [ ]:
if not os.path.exists(EVAL_CONFIG["checkpoint_path"]):
    raise FileNotFoundError(f"Không thấy {EVAL_CONFIG['checkpoint_path']}. Hãy chạy 05_train trước.")

agent = DQNAgent(OBS_DIM, N_ACTIONS, hidden_units=EVAL_CONFIG["hidden_units"])
agent.load(EVAL_CONFIG["checkpoint_path"])
print("Đã tải", EVAL_CONFIG["checkpoint_path"])

## 3. Đánh giá cuối

Agent chơi 100 ván với ε = 0, tức luôn chọn hành động có Q lớn nhất. Mỗi ván dừng ở 1.000 ống để agent giỏi không chơi mãi. Trường hợp xấu nhất (mọi ván đều chạm giới hạn) mất khoảng 18 phút.

Báo cáo trung bình, trung vị, thấp nhất và cao nhất, vì điểm dao động mạnh giữa các ván. Mốc so sánh:

| Cách chơi | Điểm trung bình |
|---|---|
| Ngẫu nhiên, vỗ cánh ở một nửa số khung hình | 0 |
| Luật viết tay (mục 1.6 của `Problem_Definition.md`) | 34,2 |

In [ ]:
scores = play_episodes(agent.greedy_action, EVAL_CONFIG["n_episodes"],
                       EVAL_CONFIG["eval_seed"], EVAL_CONFIG["score_limit"])
stats = summarize_scores(scores)
print(f"Trung bình {stats['mean']:.2f} | trung vị {stats['median']:.0f} | "
      f"thấp nhất {stats['min']} | cao nhất {stats['max']}")
print(f"Số ván chạm giới hạn {EVAL_CONFIG['score_limit']}: "
      f"{int(np.sum(scores >= EVAL_CONFIG['score_limit']))}/{len(scores)}")

plt.hist(scores, bins=20)
plt.axvline(34.2, color="gray", linestyle=":", label="luật viết tay (34,2)")
plt.xlabel("số ống vượt được")
plt.ylabel("số ván")
plt.title("Phân bố điểm của agent")
plt.legend()
plt.show()

## 4. Kiểm tra giá trị Q

Với γ = 0,99, một agent không bao giờ chết nhận tổng reward có chiết khấu khoảng 11,9 đến 12,9. Vì vậy:

- Q ở các trạng thái an toàn nên nằm quanh khoảng này.
- Trạng thái sát va chạm có Q thấp hơn, xuống tới khoảng −1.
- Không trạng thái nào có giá trị thật cao hơn 12,9. Q vượt rõ mức này là dấu hiệu đánh giá quá cao.

Ô dưới cho agent chơi một ván (tối đa 50 ống), ghi lại mọi observation, rồi xem phân bố max Q trên các trạng thái đó.

In [ ]:
def collect_states(policy, seed, score_limit):
    env = FlappyEnv(score_limit=score_limit)
    states = []
    try:
        state = env.reset(seed=seed)
        episode_over = False
        while not episode_over:
            states.append(state)
            state, _, terminated, truncated, _ = env.step(policy(state))
            episode_over = terminated or truncated
    finally:
        env.close()
    return np.array(states)


visited = collect_states(agent.greedy_action, EVAL_CONFIG["eval_seed"], score_limit=50)
max_q = agent.q_values(visited).max(axis=1)
print(f"{len(visited)} trạng thái | max Q: trung bình {max_q.mean():.2f}, "
      f"thấp nhất {max_q.min():.2f}, cao nhất {max_q.max():.2f}")
print(f"Tỷ lệ trạng thái có max Q > 12,9: {np.mean(max_q > 12.9):.1%}")

plt.hist(max_q, bins=40)
plt.axvline(12.9, color="red", linestyle="--", label="giới hạn giá trị thật (12,9)")
plt.xlabel("max Q")
plt.ylabel("số trạng thái")
plt.legend()
plt.show()

## 5. Xem agent chơi (tùy chọn)

Mở một cửa sổ pygame và cho agent chơi một ván, dừng ở 20 ống. Chế độ `"human"` chạy 30 khung hình/giây, nên 20 ống mất khoảng 25 giây.

In [ ]:
watch_score = 0
env = FlappyEnv(score_limit=20, render_mode="human")
try:
    state = env.reset(seed=EVAL_CONFIG["eval_seed"])
    episode_over = False
    while not episode_over:
        state, _, terminated, truncated, watch_score = env.step(agent.greedy_action(state))
        episode_over = terminated or truncated
finally:
    env.close()
print("Số ống vượt được:", watch_score)